# Module 2 — Data Exploration

Pull small samples of each data layer and print the head, just to see what the raw shapes look like before designing the real pipeline.

1. Raw scientific text — arXiv cond-mat abstracts (free API, no key)
2. Structured material records — Materials Project (needs `MP_API_KEY` env var; falls back to a tiny embedded sample if not set)
3. SFT instruction examples — templated from the structured records
4. DPO preference pairs — grounded vs hallucinated, templated from the structured records

In [ ]:
import os
import json
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET

DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Raw scientific text — arXiv abstracts

Query the arXiv API for `cond-mat.mtrl-sci` abstracts. No API key required.

In [ ]:
def fetch_arxiv_abstracts(query="cat:cond-mat.mtrl-sci", max_results=20):
    url = "http://export.arxiv.org/api/query"
    params = {
        "search_query": query,
        "start": 0,
        "max_results": max_results,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    ns = {"atom": "http://www.w3.org/2005/Atom"}
    root = ET.fromstring(resp.text)
    records = []
    for entry in root.findall("atom:entry", ns):
        records.append({
            "id": entry.find("atom:id", ns).text,
            "title": entry.find("atom:title", ns).text.strip().replace("\n", " "),
            "summary": entry.find("atom:summary", ns).text.strip().replace("\n", " "),
            "published": entry.find("atom:published", ns).text,
        })
    return pd.DataFrame(records)

df_text = fetch_arxiv_abstracts()
df_text.to_json(f"{DATA_DIR}/raw_text_arxiv.jsonl", orient="records", lines=True)
print(df_text.shape)
df_text.head()

## 2. Structured material records — Materials Project

Requires a free API key from https://materialsproject.org/api (set `MP_API_KEY` in your environment). If no key is set, falls back to a small embedded sample so the rest of the notebook still runs.

In [ ]:
MP_API_KEY = os.environ.get("MP_API_KEY")

FALLBACK_RECORDS = [
    {"material_id": "mp-149", "formula": "Si", "crystal_system": "cubic", "band_gap": 0.61, "formation_energy_per_atom": -0.51, "is_stable": True},
    {"material_id": "mp-2534", "formula": "GaAs", "crystal_system": "cubic", "band_gap": 0.19, "formation_energy_per_atom": -0.21, "is_stable": True},
    {"material_id": "mp-1265", "formula": "ZnO", "crystal_system": "hexagonal", "band_gap": 0.78, "formation_energy_per_atom": -1.5, "is_stable": True},
    {"material_id": "mp-22862", "formula": "NaCl", "crystal_system": "cubic", "band_gap": 4.85, "formation_energy_per_atom": -2.0, "is_stable": True},
    {"material_id": "mp-19017", "formula": "Fe2O3", "crystal_system": "trigonal", "band_gap": 0.0, "formation_energy_per_atom": -1.6, "is_stable": True},
]

def fetch_materials_project(limit=20):
    from mp_api.client import MPRester  # requires `pip install mp-api`
    with MPRester(MP_API_KEY) as mpr:
        docs = mpr.materials.summary.search(
            num_chunks=1, chunk_size=limit,
            fields=["material_id", "formula_pretty", "symmetry", "band_gap", "formation_energy_per_atom", "is_stable"],
        )
    records = [{
        "material_id": str(d.material_id),
        "formula": d.formula_pretty,
        "crystal_system": str(d.symmetry.crystal_system) if d.symmetry else None,
        "band_gap": d.band_gap,
        "formation_energy_per_atom": d.formation_energy_per_atom,
        "is_stable": d.is_stable,
    } for d in docs]
    return pd.DataFrame(records)

if MP_API_KEY:
    try:
        df_records = fetch_materials_project()
    except Exception as e:
        print(f"Materials Project fetch failed ({e}), using fallback sample")
        df_records = pd.DataFrame(FALLBACK_RECORDS)
else:
    print("MP_API_KEY not set, using fallback sample")
    df_records = pd.DataFrame(FALLBACK_RECORDS)

df_records.to_json(f"{DATA_DIR}/structured_records.jsonl", orient="records", lines=True)
print(df_records.shape)
df_records.head()

## 3. SFT instruction examples

Templated question/answer pairs generated directly from the structured records.

In [ ]:
def make_sft_examples(df):
    examples = []
    for _, r in df.iterrows():
        examples.append({
            "instruction": f"What is the band gap of {r['formula']}?",
            "response": f"{r['formula']} has a band gap of {r['band_gap']:.2f} eV, based on its {r['crystal_system']} crystal structure.",
        })
        examples.append({
            "instruction": f"Is {r['formula']} thermodynamically stable?",
            "response": f"{r['formula']} is {'stable' if r['is_stable'] else 'not stable'}, with a formation energy of {r['formation_energy_per_atom']:.2f} eV/atom.",
        })
    return pd.DataFrame(examples)

df_sft = make_sft_examples(df_records)
df_sft.to_json(f"{DATA_DIR}/sft_examples.jsonl", orient="records", lines=True)
print(df_sft.shape)
df_sft.head()

## 4. DPO preference pairs

Chosen = grounded in the record. Rejected = a plausible-sounding but unsupported/hallucinated claim.

In [ ]:
def make_dpo_pairs(df):
    pairs = []
    for _, r in df.iterrows():
        prompt = f"What is the band gap of {r['formula']} and is it stable?"
        chosen = (
            f"Based on the available data, {r['formula']} has a band gap of {r['band_gap']:.2f} eV "
            f"and a formation energy of {r['formation_energy_per_atom']:.2f} eV/atom, so it is "
            f"{'considered stable' if r['is_stable'] else 'not considered stable'}."
        )
        rejected = (
            f"{r['formula']} is a well-known high-performance material with an exceptionally large "
            f"band gap, making it ideal for next-generation power electronics."
        )
        pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    return pd.DataFrame(pairs)

df_dpo = make_dpo_pairs(df_records)
df_dpo.to_json(f"{DATA_DIR}/dpo_pairs.jsonl", orient="records", lines=True)
print(df_dpo.shape)
df_dpo.head()